# 08 — Validation-gated robust residual hybrid

This notebook implements the proposed **robust residual hybrid strategy**. A selected SARIMA model provides the correction branch's base forecast. Nonlinear regressors learn its out-of-sample residuals, and a validation-only multi-window selector decides whether a scaled correction is sufficiently stable for each resolution × target × horizon task.

The operational strategy is deliberately conservative:

1. fit the SARIMA correction base without using test data;
2. generate out-of-fold SARIMA residuals on the training partition;
3. train and tune four nonlinear residual correctors;
4. evaluate each corrector–alpha pair in four chronological validation subwindows;
5. activate the highest-scoring eligible correction, or use `BASE_ONLY` otherwise;
6. refit the selected correctors using training OOF residuals plus validation residuals and evaluate test once.

`BASE_ONLY` is explicitly defined here as the **best advanced-traditional model selected in notebook 07**, whereas the correction branch always starts from `SARIMA_DAILY_DIFFERENCE`. This distinction removes ambiguity from the operational strategy.


## Dependency note

The full experiment requires `statsmodels`, `pyarrow`, `xgboost`, and `tqdm`. If necessary, run the following command in a separate Jupyter cell, restart the kernel, and execute the notebook again:

```python
%pip install "statsmodels>=0.14,<0.15" "pyarrow>=15" "xgboost>=2.0,<4" tqdm
```


In [ ]:
from pathlib import Path
import importlib.util
import json
import math
import platform
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from statsmodels.tsa.arima.model import ARIMA

XGBOOST_AVAILABLE = importlib.util.find_spec("xgboost") is not None
TQDM_AVAILABLE = importlib.util.find_spec("tqdm") is not None
if not TQDM_AVAILABLE:
    raise ImportError(
        "This notebook requires tqdm for progress reporting. Install it with "
        "`%pip install tqdm`, restart the kernel, and rerun this notebook."
    )
from tqdm.auto import tqdm

if XGBOOST_AVAILABLE:
    from xgboost import XGBRegressor

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "resolutions").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run notebook 03 first and keep the standard folder structure."
    )


PROJECT_ROOT = find_project_root()
RESOLUTION_DIR = PROJECT_ROOT / "data" / "processed" / "resolutions"
INDEX_DIR = PROJECT_ROOT / "data" / "processed" / "effective_indices"
ADVANCED_RESULTS_DIR = PROJECT_ROOT / "results" / "advanced_traditional"
RESULTS_DIR = PROJECT_ROOT / "results" / "robust_residual_hybrid"
PREDICTION_DIR = RESULTS_DIR / "predictions"
MODEL_DIR = PROJECT_ROOT / "models" / "robust_residual_hybrid"
FIGURE_DIR = PROJECT_ROOT / "figures" / "robust_residual_hybrid"
METADATA_DIR = PROJECT_ROOT / "metadata"

for directory in [RESULTS_DIR, PREDICTION_DIR, MODEL_DIR, FIGURE_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {platform.python_version()} | Platform: {platform.platform()}")


## Experimental configuration

`EXECUTION_MODE = "full"` is mandatory for scientific results. `"smoke_test"` limits the experiment to the 60-minute resolution, two correctors, their first candidates, and 20 validation/test origins. Smoke-test thresholds are relaxed only to exercise the selector and must never be reported.

The historical 4-minute validation experiment used 300 evenly spaced origins, providing 75 observations in each of four windows. The other resolutions use all valid validation origins.


In [ ]:
EXECUTION_MODE = "full"  # Use "smoke_test" only for a technical pipeline check.

RESOLUTIONS = [4, 12, 20, 30, 60]
TARGETS = ["temperature", "relative_humidity"]
HORIZONS_MINUTES = [60, 120, 240, 480]
HISTORY_HOURS = 24
CORRECTION_BASE_FAMILY = "SARIMA_DAILY_DIFFERENCE"
FALLBACK_POLICY = "BEST_ADVANCED_TRADITIONAL"
CORRECTORS = ["HIST_GRADIENT_BOOSTING", "EXTRA_TREES", "XGBOOST", "RANDOM_FOREST"]
ALPHA_GRID = [0.25, 0.50, 0.75, 1.00]
RANDOM_SEED = 2026
OOF_FOLDS = 4
OOF_INITIAL_TRAIN_FRACTION = 0.25
VALIDATION_WINDOWS = 4
MINIMUM_SAMPLES_PER_WINDOW = 10
MINIMUM_POSITIVE_WINDOW_FRACTION = 0.75
MINIMUM_MEDIAN_GAIN_PCT = 0.5
MINIMUM_WORST_WINDOW_GAIN_PCT = -1.0
IQR_PENALTY_WEIGHT = 0.50
WORST_LOSS_PENALTY_WEIGHT = 0.25
TIE_TOLERANCE = 1e-6
VALIDATION_ORIGIN_CAP = {4: 300, 12: None, 20: None, 30: None, 60: None}

LAG_HOURS = [0, 1, 2, 3, 6, 12, 24]
DELTA_HOURS = [1, 2, 3, 6, 12, 24]
ROLLING_HOURS = [1, 3, 6, 12, 24]
HISTORY_SOURCES = [
    "temperature", "relative_humidity", "temperature_bme280",
    "relative_humidity_bme280", "pressure", "rh_bme280_saturated_fraction",
]
STATIC_FEATURES = ["hour_sin", "hour_cos", "day_of_year_sin", "day_of_year_cos"]

CORRECTOR_GRID = {
    "HIST_GRADIENT_BOOSTING": [
        {"candidate_id": 1, "max_iter": 180, "learning_rate": 0.05, "max_leaf_nodes": 31, "l2_regularization": 0.1},
        {"candidate_id": 2, "max_iter": 260, "learning_rate": 0.04, "max_leaf_nodes": 63, "l2_regularization": 1.0},
        {"candidate_id": 3, "max_iter": 180, "learning_rate": 0.08, "max_leaf_nodes": 31, "l2_regularization": 5.0},
    ],
    "EXTRA_TREES": [
        {"candidate_id": 1, "n_estimators": 250, "max_depth": None, "min_samples_leaf": 1, "max_features": 0.7},
        {"candidate_id": 2, "n_estimators": 320, "max_depth": 20, "min_samples_leaf": 2, "max_features": 0.85},
        {"candidate_id": 3, "n_estimators": 320, "max_depth": 14, "min_samples_leaf": 4, "max_features": 1.0},
    ],
    "XGBOOST": [
        {"candidate_id": 1, "n_estimators": 260, "learning_rate": 0.04, "max_depth": 5, "subsample": 0.85, "colsample_bytree": 0.85, "reg_lambda": 1.0, "min_child_weight": 1},
        {"candidate_id": 2, "n_estimators": 360, "learning_rate": 0.03, "max_depth": 6, "subsample": 0.85, "colsample_bytree": 0.80, "reg_lambda": 3.0, "min_child_weight": 2},
        {"candidate_id": 3, "n_estimators": 220, "learning_rate": 0.07, "max_depth": 4, "subsample": 0.90, "colsample_bytree": 0.90, "reg_lambda": 5.0, "min_child_weight": 3},
    ],
    "RANDOM_FOREST": [
        {"candidate_id": 1, "n_estimators": 240, "max_depth": None, "min_samples_leaf": 1, "max_features": 0.7},
        {"candidate_id": 2, "n_estimators": 300, "max_depth": 20, "min_samples_leaf": 2, "max_features": 0.85},
        {"candidate_id": 3, "n_estimators": 300, "max_depth": 14, "min_samples_leaf": 4, "max_features": 1.0},
    ],
}

if EXECUTION_MODE == "full":
    ACTIVE_RESOLUTIONS = RESOLUTIONS
    ACTIVE_CORRECTORS = CORRECTORS
    ACTIVE_GRID = CORRECTOR_GRID
    ACTIVE_MINIMUM_SAMPLES_PER_WINDOW = MINIMUM_SAMPLES_PER_WINDOW
    MAX_ORIGINS_PER_EVALUATION_SPLIT = None
    if not XGBOOST_AVAILABLE:
        raise ImportError(
            "Full mode requires xgboost. Install it with `%pip install \"xgboost>=2.0,<4\"`, "
            "restart the kernel, and rerun this notebook."
        )
elif EXECUTION_MODE == "smoke_test":
    ACTIVE_RESOLUTIONS = [60]
    ACTIVE_CORRECTORS = ["HIST_GRADIENT_BOOSTING", "EXTRA_TREES"]
    ACTIVE_GRID = {name: CORRECTOR_GRID[name][:1] for name in ACTIVE_CORRECTORS}
    ACTIVE_MINIMUM_SAMPLES_PER_WINDOW = 3
    MAX_ORIGINS_PER_EVALUATION_SPLIT = 20
else:
    raise ValueError("EXECUTION_MODE must be 'full' or 'smoke_test'.")

configuration = {
    "execution_mode": EXECUTION_MODE,
    "temporal_resolutions_minutes": RESOLUTIONS,
    "targets": TARGETS,
    "forecast_horizons_minutes": HORIZONS_MINUTES,
    "history_hours": HISTORY_HOURS,
    "correction_base_family": CORRECTION_BASE_FAMILY,
    "fallback_policy": FALLBACK_POLICY,
    "correctors": CORRECTORS,
    "corrector_candidates": CORRECTOR_GRID,
    "alpha_grid": ALPHA_GRID,
    "oof_training": {"method": "blocked expanding origin", "folds": OOF_FOLDS, "initial_train_fraction": OOF_INITIAL_TRAIN_FRACTION},
    "validation_windows": VALIDATION_WINDOWS,
    "minimum_samples_per_window": MINIMUM_SAMPLES_PER_WINDOW,
    "eligibility_rule": {
        "minimum_positive_window_fraction": MINIMUM_POSITIVE_WINDOW_FRACTION,
        "minimum_median_gain_pct": MINIMUM_MEDIAN_GAIN_PCT,
        "minimum_worst_window_gain_pct": MINIMUM_WORST_WINDOW_GAIN_PCT,
    },
    "robustness_score": "median_gain - 0.50*IQR(gain) - 0.25*max(0, -worst_gain)",
    "validation_origin_cap": VALIDATION_ORIGIN_CAP,
    "progress_reporting": {
        "library": "tqdm.auto",
        "levels": [
            "SARIMA task and forecast origin",
            "corrector candidate",
            "corrector-alpha robustness candidate",
            "final corrector refit",
        ],
    },
    "test_policy": "Test is never used to select the base, corrector, alpha, or activation decision.",
    "random_seed": RANDOM_SEED,
}
(METADATA_DIR / "08_robust_residual_hybrid_configuration.json").write_text(
    json.dumps(configuration, indent=2), encoding="utf-8"
)
configuration


## Input audit

Notebook 07 must have completed in the same execution mode. Its all-candidate validation predictions identify the SARIMA correction base, while its selected advanced-traditional validation/test predictions implement `BASE_ONLY`.


In [ ]:
EXPECTED_COUNTS = {
    4: {"train": 1053, "validation": 1446, "test": 2742},
    12: {"train": 104, "validation": 135, "test": 378},
    20: {"train": 146, "validation": 239, "test": 556},
    30: {"train": 104, "validation": 142, "test": 350},
    60: {"train": 66, "validation": 58, "test": 211},
}

ADVANCED_CONFIG_FILE = METADATA_DIR / "07_advanced_traditional_configuration.json"
ADVANCED_SELECTION_FILE = ADVANCED_RESULTS_DIR / "07_selected_configuration_by_resolution_target.csv"
ADVANCED_VALIDATION_METRICS_FILE = ADVANCED_RESULTS_DIR / "04_candidate_validation_metrics_by_horizon.csv"
ADVANCED_VALIDATION_PREDICTIONS_FILE = ADVANCED_RESULTS_DIR / "predictions" / "01_all_candidate_validation_predictions.parquet"
ADVANCED_TEST_PREDICTIONS_FILE = ADVANCED_RESULTS_DIR / "predictions" / "02_selected_advanced_traditional_test_predictions.parquet"

required_files = [
    ADVANCED_CONFIG_FILE, ADVANCED_SELECTION_FILE, ADVANCED_VALIDATION_METRICS_FILE,
    ADVANCED_VALIDATION_PREDICTIONS_FILE, ADVANCED_TEST_PREDICTIONS_FILE,
]
missing = [str(path.relative_to(PROJECT_ROOT)) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("Run notebook 07 first. Missing: " + ", ".join(missing))

advanced_configuration = json.loads(ADVANCED_CONFIG_FILE.read_text(encoding="utf-8"))
if advanced_configuration["execution_mode"] != EXECUTION_MODE:
    raise RuntimeError(
        f"Notebook 07 was run in {advanced_configuration['execution_mode']!r}, but notebook 08 is "
        f"configured as {EXECUTION_MODE!r}. Use matching execution modes."
    )

datasets = {}
origins_by_resolution = {}
audit_rows = []
for resolution in RESOLUTIONS:
    data = pd.read_csv(
        RESOLUTION_DIR / f"greenhouse_{resolution}min.csv", parse_dates=["timestamp"]
    ).sort_values("timestamp").reset_index(drop=True)
    origins = pd.read_csv(
        INDEX_DIR / f"effective_indices_{resolution}min.csv", parse_dates=["origin_timestamp"]
    )
    datasets[resolution] = data
    origins_by_resolution[resolution] = origins
    row = {"resolution": f"{resolution}min", "dataset_rows": len(data)}
    for split, expected in EXPECTED_COUNTS[resolution].items():
        observed = int(origins["split"].eq(split).sum())
        row[f"{split}_origins"] = observed
        if observed != expected:
            raise AssertionError(f"Unexpected {resolution} min {split} count: {observed} != {expected}")
    audit_rows.append(row)

sample_audit = pd.DataFrame(audit_rows)
sample_audit.to_csv(RESULTS_DIR / "01_sample_audit.csv", index=False)
display(sample_audit)


## Select the SARIMA correction base and load `BASE_ONLY`

The correction base is the best `SARIMA_DAILY_DIFFERENCE` candidate for each resolution and target according to notebook 07 validation metrics. It need not be the overall advanced-traditional winner. The latter is retained separately as the operational fallback.


In [ ]:
advanced_validation_metrics = pd.read_csv(ADVANCED_VALIDATION_METRICS_FILE)
advanced_selection = pd.read_csv(ADVANCED_SELECTION_FILE)
advanced_validation_predictions = pd.read_parquet(ADVANCED_VALIDATION_PREDICTIONS_FILE)
advanced_test_predictions = pd.read_parquet(ADVANCED_TEST_PREDICTIONS_FILE)

sarima_summary = (
    advanced_validation_metrics.loc[
        advanced_validation_metrics["family"].eq(CORRECTION_BASE_FAMILY)
    ]
    .groupby(
        ["resolution", "resolution_minutes", "target", "family", "candidate_id", "parameters"],
        as_index=False,
    )
    .agg(mean_validation_nrmse=("nrmse", "mean"), mean_validation_r2=("r2", "mean"))
)
selected_sarima = (
    sarima_summary.sort_values(
        ["resolution_minutes", "target", "mean_validation_nrmse", "mean_validation_r2"],
        ascending=[True, True, True, False],
    )
    .groupby(["resolution_minutes", "target"], as_index=False, sort=False)
    .head(1)
    .reset_index(drop=True)
)
selected_sarima.to_csv(RESULTS_DIR / "02_selected_sarima_correction_base.csv", index=False)


def evenly_cap_origins(predictions, resolution):
    cap = VALIDATION_ORIGIN_CAP[resolution]
    if MAX_ORIGINS_PER_EVALUATION_SPLIT is not None:
        cap = MAX_ORIGINS_PER_EVALUATION_SPLIT
    unique_origins = np.sort(predictions["origin_index"].unique())
    if cap is None or len(unique_origins) <= cap:
        return predictions.copy()
    positions = np.linspace(0, len(unique_origins) - 1, cap).round().astype(int)
    keep = np.unique(unique_origins[positions])
    return predictions.loc[predictions["origin_index"].isin(keep)].copy()


base_validation_frames = []
fallback_validation_frames = []
fallback_test_frames = []
for selected in selected_sarima.itertuples(index=False):
    subset = advanced_validation_predictions.loc[
        advanced_validation_predictions["resolution_minutes"].eq(selected.resolution_minutes)
        & advanced_validation_predictions["target"].eq(selected.target)
        & advanced_validation_predictions["family"].eq(CORRECTION_BASE_FAMILY)
        & advanced_validation_predictions["candidate_id"].eq(selected.candidate_id)
    ]
    base_validation_frames.append(evenly_cap_origins(subset, int(selected.resolution_minutes)))

for selected in advanced_selection.itertuples(index=False):
    subset = advanced_validation_predictions.loc[
        advanced_validation_predictions["resolution_minutes"].eq(selected.resolution_minutes)
        & advanced_validation_predictions["target"].eq(selected.target)
        & advanced_validation_predictions["family"].eq(selected.family)
        & advanced_validation_predictions["candidate_id"].eq(selected.candidate_id)
    ]
    fallback_validation_frames.append(evenly_cap_origins(subset, int(selected.resolution_minutes)))
    fallback_test_frames.append(advanced_test_predictions.loc[
        advanced_test_predictions["resolution_minutes"].eq(selected.resolution_minutes)
        & advanced_test_predictions["target"].eq(selected.target)
    ].copy())

base_validation_predictions = pd.concat(base_validation_frames, ignore_index=True)
fallback_validation_predictions = pd.concat(fallback_validation_frames, ignore_index=True)
fallback_test_predictions = pd.concat(fallback_test_frames, ignore_index=True)

if EXECUTION_MODE == "smoke_test":
    base_validation_predictions = base_validation_predictions.loc[
        base_validation_predictions["resolution_minutes"].isin(ACTIVE_RESOLUTIONS)
    ].copy()
    fallback_validation_predictions = fallback_validation_predictions.loc[
        fallback_validation_predictions["resolution_minutes"].isin(ACTIVE_RESOLUTIONS)
    ].copy()
    fallback_test_predictions = fallback_test_predictions.loc[
        fallback_test_predictions["resolution_minutes"].isin(ACTIVE_RESOLUTIONS)
    ].copy()

display(selected_sarima)


## Out-of-fold SARIMA residuals

Training residuals are generated with four blocked expanding-origin folds. The first 25% of the eligible training origins initializes the first fit; predictions are then made for the remaining origins in four chronological blocks. No corrector is trained on an in-sample SARIMA residual.


In [ ]:
def causal_target_values(data, target, fitting_splits):
    values = data[target].astype(float).ffill()
    median = float(values.loc[data["split"].isin(fitting_splits)].median())
    return values.fillna(median).to_numpy(dtype=float)


def fit_daily_sarima(values, parameters, seasonal_steps):
    differenced = values[seasonal_steps:] - values[:-seasonal_steps]
    started = time.perf_counter()
    result = ARIMA(
        differenced, order=tuple(parameters["order"]), trend=parameters["trend"]
    ).fit()
    return result, time.perf_counter() - started


def daily_sarima_forecast(result, history, seasonal_steps, maximum_steps):
    differenced = history[seasonal_steps:] - history[:-seasonal_steps]
    applied = result.apply(differenced, refit=False)
    difference_forecast = np.asarray(applied.forecast(maximum_steps), dtype=float)
    reconstructed = list(history)
    forecast = []
    for difference in difference_forecast:
        value = reconstructed[-seasonal_steps] + difference
        reconstructed.append(float(value))
        forecast.append(float(value))
    return np.asarray(forecast)


def forecast_sarima_origins(
    result, data, origins, values, resolution, target, split_label,
    progress_description=None,
):
    seasonal_steps = 24 * 60 // resolution
    maximum_steps = max(HORIZONS_MINUTES) // resolution
    rows = []
    origin_iterator = origins.itertuples(index=False)
    if progress_description is not None:
        origin_iterator = tqdm(
            origin_iterator,
            total=len(origins),
            desc=progress_description,
            unit="origin",
            leave=False,
            position=1,
            dynamic_ncols=True,
        )
    for origin in origin_iterator:
        start = max(0, int(origin.history_start_index) - seasonal_steps)
        end = int(origin.history_end_index) + 1
        forecast = daily_sarima_forecast(result, values[start:end], seasonal_steps, maximum_steps)
        for horizon in HORIZONS_MINUTES:
            steps = horizon // resolution
            target_index = int(origin.origin_index) + steps
            rows.append({
                "resolution": f"{resolution}min",
                "resolution_minutes": resolution,
                "target": target,
                "horizon_minutes": horizon,
                "split": split_label,
                "origin_index": int(origin.origin_index),
                "origin_timestamp": origin.origin_timestamp,
                "target_timestamp": data.loc[target_index, "timestamp"],
                "observed": float(values[target_index]),
                "predicted": float(forecast[steps - 1]),
            })
    return pd.DataFrame(rows)


oof_prediction_frames = []
oof_fit_rows = []
test_base_frames = []
test_base_fit_rows = []

selected_sarima_tasks = list(selected_sarima.loc[
    selected_sarima["resolution_minutes"].isin(ACTIVE_RESOLUTIONS)
].itertuples(index=False))

for selected in tqdm(
    selected_sarima_tasks,
    desc="Phase A — OOF SARIMA and test base",
    unit="task",
    dynamic_ncols=True,
):
    resolution = int(selected.resolution_minutes)
    target = selected.target
    parameters = json.loads(selected.parameters)
    data = datasets[resolution]
    origins = origins_by_resolution[resolution]
    train_origins = origins.loc[origins["split"].eq("train")].reset_index(drop=True)
    values = causal_target_values(data, target, ["train"])
    seasonal_steps = 24 * 60 // resolution

    initial_count = int(math.ceil(len(train_origins) * OOF_INITIAL_TRAIN_FRACTION))
    fold_positions = np.array_split(np.arange(initial_count, len(train_origins)), OOF_FOLDS)
    for fold_id, positions in enumerate(fold_positions, start=1):
        if not len(positions):
            continue
        fold_origins = train_origins.iloc[positions].copy()
        fit_end_index = int(train_origins.iloc[int(positions[0]) - 1]["origin_index"])
        result, fit_seconds = fit_daily_sarima(
            values[:fit_end_index + 1], parameters, seasonal_steps
        )
        predictions = forecast_sarima_origins(
            result, data, fold_origins, values, resolution, target, "train_oof",
            progress_description=(
                f"{resolution}min {target} — OOF fold {fold_id}/{OOF_FOLDS}"
            ),
        )
        predictions["fold_id"] = fold_id
        oof_prediction_frames.append(predictions)
        oof_fit_rows.append({
            "resolution": f"{resolution}min", "target": target, "fold_id": fold_id,
            "fit_end_index": fit_end_index,
            "fit_end_timestamp": data.loc[fit_end_index, "timestamp"],
            "n_origins": len(fold_origins), "parameters": selected.parameters,
            "aic": float(result.aic), "bic": float(result.bic), "fit_seconds": fit_seconds,
        })

    test_origin_ids = fallback_test_predictions.loc[
        fallback_test_predictions["resolution_minutes"].eq(resolution)
        & fallback_test_predictions["target"].eq(target), "origin_index"
    ].unique()
    test_origins = origins.loc[origins["origin_index"].isin(test_origin_ids)].copy()
    final_values = causal_target_values(data, target, ["train", "validation"])
    validation_rows = np.flatnonzero(data["split"].eq("validation").to_numpy())
    final_fit_end = int(validation_rows[-1])
    final_result, fit_seconds = fit_daily_sarima(
        final_values[:final_fit_end + 1], parameters, seasonal_steps
    )
    test_base_frames.append(forecast_sarima_origins(
        final_result, data, test_origins, final_values, resolution, target, "test",
        progress_description=f"{resolution}min {target} — test base",
    ))
    test_base_fit_rows.append({
        "resolution": f"{resolution}min", "target": target,
        "fit_end_index": final_fit_end, "parameters": selected.parameters,
        "aic": float(final_result.aic), "bic": float(final_result.bic),
        "fit_seconds": fit_seconds,
    })

oof_base_predictions = pd.concat(oof_prediction_frames, ignore_index=True)
test_base_predictions = pd.concat(test_base_frames, ignore_index=True)
oof_base_predictions.to_parquet(PREDICTION_DIR / "01_oof_sarima_base_predictions.parquet", index=False)
test_base_predictions.to_parquet(PREDICTION_DIR / "02_sarima_base_test_predictions.parquet", index=False)
pd.DataFrame(oof_fit_rows).to_csv(RESULTS_DIR / "03_oof_sarima_fit_summary.csv", index=False)
pd.DataFrame(test_base_fit_rows).to_csv(RESULTS_DIR / "04_sarima_base_test_fit_summary.csv", index=False)


## Residual-corrector features

Each origin is represented by 206 features: lag, difference, and rolling summaries for six sensor-derived sources; four cyclic time variables; and the four SARIMA base forecasts. Missing sensor features are imputed from the corrector's fitting data only.


In [ ]:
def safe_value(values, index):
    return float(values[index]) if 0 <= index < len(values) else np.nan


def summarize_source(values, origin, resolution, source):
    features = {}
    for hours in LAG_HOURS:
        steps = hours * 60 // resolution
        features[f"{source}__lag_{hours}h"] = safe_value(values, origin - steps)
    for hours in DELTA_HOURS:
        steps = hours * 60 // resolution
        current = safe_value(values, origin)
        lagged = safe_value(values, origin - steps)
        features[f"{source}__delta_{hours}h"] = current - lagged
    for hours in ROLLING_HOURS:
        steps = hours * 60 // resolution
        start = max(0, origin - steps + 1)
        window = np.asarray(values[start:origin + 1], dtype=float)
        finite = window[np.isfinite(window)]
        for statistic in ["mean", "std", "min", "max"]:
            name = f"{source}__roll_{statistic}_{hours}h"
            if not len(finite):
                features[name] = np.nan
            elif statistic == "mean":
                features[name] = float(np.mean(finite))
            elif statistic == "std":
                features[name] = float(np.std(finite, ddof=0))
            elif statistic == "min":
                features[name] = float(np.min(finite))
            else:
                features[name] = float(np.max(finite))
    return features


def build_corrector_dataset(data, origins, base_predictions, resolution, target):
    target_base = base_predictions.loc[
        base_predictions["resolution_minutes"].eq(resolution)
        & base_predictions["target"].eq(target)
    ].copy()
    origin_ids = np.sort(target_base["origin_index"].unique())
    origin_table = origins.loc[origins["origin_index"].isin(origin_ids)].copy()
    origin_table = origin_table.sort_values("origin_index").reset_index(drop=True)
    base_wide = target_base.pivot(index="origin_index", columns="horizon_minutes", values="predicted")
    observed_wide = target_base.pivot(index="origin_index", columns="horizon_minutes", values="observed")

    source_values = {source: data[source].to_numpy(dtype=float) for source in HISTORY_SOURCES}
    feature_rows = []
    for row in origin_table.itertuples(index=False):
        origin = int(row.origin_index)
        features = {"origin_index": origin, "origin_timestamp": row.origin_timestamp}
        for source, values in source_values.items():
            features.update(summarize_source(values, origin, resolution, source))
        for static in STATIC_FEATURES:
            features[f"{static}__origin"] = float(data.at[origin, static])
        for horizon in HORIZONS_MINUTES:
            features[f"base_forecast_h{horizon}"] = float(base_wide.at[origin, horizon])
        feature_rows.append(features)

    feature_frame = pd.DataFrame(feature_rows).sort_values("origin_index").reset_index(drop=True)
    feature_columns = sorted(set(feature_frame.columns) - {"origin_index", "origin_timestamp"})
    residuals = np.column_stack([
        observed_wide.loc[feature_frame["origin_index"], horizon].to_numpy(dtype=float)
        - base_wide.loc[feature_frame["origin_index"], horizon].to_numpy(dtype=float)
        for horizon in HORIZONS_MINUTES
    ])
    return feature_frame, feature_columns, residuals


feature_catalog_rows = []
example_resolution = ACTIVE_RESOLUTIONS[0]
example_target = TARGETS[0]
example_features, FEATURE_COLUMNS, _ = build_corrector_dataset(
    datasets[example_resolution], origins_by_resolution[example_resolution],
    oof_base_predictions, example_resolution, example_target,
)
assert len(FEATURE_COLUMNS) == 206, f"Expected 206 corrector features, found {len(FEATURE_COLUMNS)}"
for feature in FEATURE_COLUMNS:
    feature_catalog_rows.append({"feature_name": feature})
pd.DataFrame(feature_catalog_rows).to_csv(RESULTS_DIR / "05_corrector_feature_catalog.csv", index=False)
print(f"Corrector feature count: {len(FEATURE_COLUMNS)}")


## Corrector tuning on validation

Each candidate predicts the four residual horizons simultaneously. Candidate selection is performed independently within each corrector family, resolution, and target using mean validation NRMSE of the fully corrected forecast (`alpha = 1`). This step retains all four corrector families for the subsequent robustness selector.


In [ ]:
def make_corrector(name, parameters):
    parameters = {k: v for k, v in parameters.items() if k != "candidate_id"}
    if name == "HIST_GRADIENT_BOOSTING":
        estimator = MultiOutputRegressor(
            HistGradientBoostingRegressor(random_state=RANDOM_SEED, **parameters), n_jobs=-1
        )
    elif name == "EXTRA_TREES":
        estimator = ExtraTreesRegressor(random_state=RANDOM_SEED, n_jobs=-1, **parameters)
    elif name == "RANDOM_FOREST":
        estimator = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1, **parameters)
    elif name == "XGBOOST":
        estimator = MultiOutputRegressor(XGBRegressor(
            random_state=RANDOM_SEED, n_jobs=-1, tree_method="hist",
            objective="reg:squarederror", **parameters,
        ), n_jobs=-1)
    else:
        raise ValueError(name)
    return make_pipeline(SimpleImputer(strategy="median"), estimator)


def prediction_scale(data, target, fitting_splits):
    values = data.loc[data["split"].isin(fitting_splits), target].to_numpy(dtype=float)
    return float(np.nanstd(values, ddof=0))


def score_multi_horizon(base_predictions, correction, scale):
    base_wide = base_predictions.pivot(index="origin_index", columns="horizon_minutes", values="predicted")
    observed_wide = base_predictions.pivot(index="origin_index", columns="horizon_minutes", values="observed")
    origin_order = np.sort(base_predictions["origin_index"].unique())
    rows = []
    for column, horizon in enumerate(HORIZONS_MINUTES):
        observed = observed_wide.loc[origin_order, horizon].to_numpy(dtype=float)
        predicted = base_wide.loc[origin_order, horizon].to_numpy(dtype=float) + correction[:, column]
        rmse = mean_squared_error(observed, predicted) ** 0.5
        rows.append({
            "horizon_minutes": horizon, "rmse": rmse,
            "nrmse": rmse / scale, "r2": r2_score(observed, predicted),
        })
    return pd.DataFrame(rows)


tuning_rows = []
selected_corrector_rows = []
validation_correction_cache = {}
corrector_datasets = {}

tuning_total = sum(
    len(ACTIVE_GRID[corrector])
    for _resolution in ACTIVE_RESOLUTIONS
    for _target in TARGETS
    for corrector in ACTIVE_CORRECTORS
)
tuning_progress = tqdm(
    total=tuning_total,
    desc="Phase B — corrector tuning",
    unit="candidate",
    dynamic_ncols=True,
)

for resolution in ACTIVE_RESOLUTIONS:
    data = datasets[resolution]
    origins = origins_by_resolution[resolution]
    for target in TARGETS:
        tuning_progress.set_postfix(
            resolution=f"{resolution}min", target=target, stage="features", refresh=True
        )
        X_oof_frame, feature_columns, Y_oof = build_corrector_dataset(
            data, origins, oof_base_predictions, resolution, target
        )
        X_validation_frame, validation_columns, Y_validation = build_corrector_dataset(
            data, origins, base_validation_predictions, resolution, target
        )
        if feature_columns != validation_columns:
            raise AssertionError("Corrector feature columns differ across splits.")
        X_oof = X_oof_frame[feature_columns].to_numpy(dtype=float)
        X_validation = X_validation_frame[feature_columns].to_numpy(dtype=float)
        validation_base = base_validation_predictions.loc[
            base_validation_predictions["resolution_minutes"].eq(resolution)
            & base_validation_predictions["target"].eq(target)
        ].copy()
        scale = prediction_scale(data, target, ["train"])
        corrector_datasets[(resolution, target)] = {
            "feature_columns": feature_columns,
            "X_oof_frame": X_oof_frame, "X_oof": X_oof, "Y_oof": Y_oof,
            "X_validation_frame": X_validation_frame,
            "X_validation": X_validation, "Y_validation": Y_validation,
            "validation_base": validation_base,
        }

        for corrector in ACTIVE_CORRECTORS:
            family_rows = []
            for candidate in ACTIVE_GRID[corrector]:
                tuning_progress.set_postfix(
                    resolution=f"{resolution}min",
                    target=target,
                    corrector=corrector,
                    candidate=candidate["candidate_id"],
                    refresh=True,
                )
                estimator = make_corrector(corrector, candidate)
                started = time.perf_counter()
                estimator.fit(X_oof, Y_oof)
                fit_seconds = time.perf_counter() - started
                started = time.perf_counter()
                correction = np.asarray(estimator.predict(X_validation), dtype=float)
                inference_seconds = time.perf_counter() - started
                metrics = score_multi_horizon(validation_base, correction, scale)
                row = {
                    "resolution": f"{resolution}min", "resolution_minutes": resolution,
                    "target": target, "corrector": corrector,
                    "candidate_id": candidate["candidate_id"],
                    "parameters": json.dumps({k: v for k, v in candidate.items() if k != "candidate_id"}, sort_keys=True),
                    "mean_validation_nrmse": metrics["nrmse"].mean(),
                    "mean_validation_rmse": metrics["rmse"].mean(),
                    "mean_validation_r2": metrics["r2"].mean(),
                    "minimum_validation_r2": metrics["r2"].min(),
                    "fit_seconds": fit_seconds, "inference_seconds": inference_seconds,
                }
                tuning_rows.append(row)
                family_rows.append((row, estimator, correction))
                tuning_progress.update(1)

            family_rows.sort(key=lambda item: (
                item[0]["mean_validation_nrmse"], -item[0]["mean_validation_r2"]
            ))
            winner, estimator, correction = family_rows[0]
            selected_corrector_rows.append({**winner, "selection_rule": "lowest mean validation NRMSE; mean validation R2 secondary"})
            validation_correction_cache[(resolution, target, corrector)] = correction

tuning_progress.close()

corrector_tuning = pd.DataFrame(tuning_rows)
selected_correctors = pd.DataFrame(selected_corrector_rows)
corrector_tuning.to_csv(RESULTS_DIR / "06_corrector_tuning.csv", index=False)
selected_correctors.to_csv(RESULTS_DIR / "07_selected_corrector_configuration.csv", index=False)
display(selected_correctors)


## Multi-window robustness selector

For each task and corrector–alpha pair, the validation gain in window (w) is

\[
G_w = 100\frac{\mathrm{RMSE}_{base,w}-\mathrm{RMSE}_{hybrid,w}}
{\mathrm{RMSE}_{base,w}}.
\]

A candidate is eligible when all four windows have at least 10 observations, at least 75% of windows have positive gain, the median gain is at least 0.5%, and the worst-window gain is at least −1.0%. Eligible candidates are ranked by

\[
S_{rob}=G_e-0.50\,IQR(G)-0.25\max(0,-G_{min}).
\]


In [ ]:
validation_corrector_frames = []
for (resolution, target), data_bundle in corrector_datasets.items():
    base = data_bundle["validation_base"].copy()
    origin_order = np.sort(base["origin_index"].unique())
    for corrector in ACTIVE_CORRECTORS:
        correction = validation_correction_cache[(resolution, target, corrector)]
        correction_wide = pd.DataFrame(
            correction, index=origin_order, columns=HORIZONS_MINUTES
        )
        subset = base.copy()
        subset["corrector"] = corrector
        subset["residual_prediction"] = [
            correction_wide.at[int(row.origin_index), int(row.horizon_minutes)]
            for row in subset.itertuples(index=False)
        ]
        validation_corrector_frames.append(subset)

validation_corrector_predictions = pd.concat(validation_corrector_frames, ignore_index=True)
validation_corrector_predictions.to_parquet(
    PREDICTION_DIR / "03_selected_corrector_validation_predictions.parquet", index=False
)

window_rows = []
robustness_rows = []
robustness_groups = validation_corrector_predictions.groupby(
    ["resolution", "resolution_minutes", "target", "horizon_minutes", "corrector"],
    sort=False, observed=True,
)
robustness_progress = tqdm(
    total=robustness_groups.ngroups * len(ALPHA_GRID),
    desc="Phase C — corrector-alpha robustness",
    unit="combination",
    dynamic_ncols=True,
)
for keys, group in robustness_groups:
    group = group.sort_values("origin_timestamp").reset_index(drop=True)
    windows = np.array_split(np.arange(len(group)), VALIDATION_WINDOWS)
    for alpha in ALPHA_GRID:
        robustness_progress.set_postfix(
            resolution=keys[0],
            target=keys[2],
            horizon=f"{keys[3]}min",
            corrector=keys[4],
            alpha=alpha,
            refresh=False,
        )
        gains = []
        window_ns = []
        for window_id, positions in enumerate(windows, start=1):
            window = group.iloc[positions]
            observed = window["observed"].to_numpy(dtype=float)
            base_predicted = window["predicted"].to_numpy(dtype=float)
            hybrid_predicted = base_predicted + alpha * window["residual_prediction"].to_numpy(dtype=float)
            base_rmse = mean_squared_error(observed, base_predicted) ** 0.5
            hybrid_rmse = mean_squared_error(observed, hybrid_predicted) ** 0.5
            gain = 100 * (base_rmse - hybrid_rmse) / base_rmse
            gains.append(float(gain))
            window_ns.append(len(window))
            window_rows.append({
                "resolution": keys[0], "resolution_minutes": keys[1],
                "target": keys[2], "horizon_minutes": keys[3], "corrector": keys[4],
                "alpha": alpha, "window_id": window_id, "n": len(window),
                "base_rmse": base_rmse, "hybrid_rmse": hybrid_rmse, "gain_pct": gain,
            })

        gains = np.asarray(gains, dtype=float)
        q25, q75 = np.quantile(gains, [0.25, 0.75])
        median_gain = float(np.median(gains))
        worst_gain = float(np.min(gains))
        positive_fraction = float(np.mean(gains > 0))
        gain_iqr = float(q75 - q25)
        worst_loss = max(0.0, -worst_gain)
        robust_score = median_gain - IQR_PENALTY_WEIGHT * gain_iqr - WORST_LOSS_PENALTY_WEIGHT * worst_loss
        passes_window_count = len(gains) == VALIDATION_WINDOWS
        passes_sample_count = min(window_ns) >= ACTIVE_MINIMUM_SAMPLES_PER_WINDOW
        passes_positive_fraction = positive_fraction >= MINIMUM_POSITIVE_WINDOW_FRACTION
        passes_median_gain = median_gain >= MINIMUM_MEDIAN_GAIN_PCT
        passes_worst_window = worst_gain >= MINIMUM_WORST_WINDOW_GAIN_PCT
        eligible = all([
            passes_window_count, passes_sample_count, passes_positive_fraction,
            passes_median_gain, passes_worst_window,
        ])
        robustness_rows.append({
            "resolution": keys[0], "resolution_minutes": keys[1],
            "target": keys[2], "horizon_minutes": keys[3], "corrector": keys[4],
            "alpha": alpha, "n_windows": len(gains), "minimum_window_n": min(window_ns),
            "mean_gain_pct": float(np.mean(gains)), "median_gain_pct": median_gain,
            "worst_window_gain_pct": worst_gain, "best_window_gain_pct": float(np.max(gains)),
            "gain_std_pct": float(np.std(gains, ddof=0)), "gain_q25_pct": float(q25),
            "gain_q75_pct": float(q75), "positive_windows": int(np.sum(gains > 0)),
            "positive_window_fraction": positive_fraction, "gain_iqr_pct": gain_iqr,
            "worst_loss_magnitude_pct": worst_loss, "robust_score": robust_score,
            "passes_window_count": passes_window_count, "passes_sample_count": passes_sample_count,
            "passes_positive_fraction": passes_positive_fraction,
            "passes_median_gain": passes_median_gain,
            "passes_worst_window": passes_worst_window, "eligible": eligible,
        })
        robustness_progress.update(1)

robustness_progress.close()

window_candidate_metrics = pd.DataFrame(window_rows)
candidate_robustness = pd.DataFrame(robustness_rows)
window_candidate_metrics.to_csv(RESULTS_DIR / "08_window_candidate_metrics.csv", index=False)
candidate_robustness.to_csv(RESULTS_DIR / "09_candidate_robustness_summary.csv", index=False)


In [ ]:
operational_rows = []
task_columns = ["resolution", "resolution_minutes", "target", "horizon_minutes"]
for keys, group in candidate_robustness.groupby(task_columns, sort=False, observed=True):
    eligible = group.loc[group["eligible"]].copy()
    if eligible.empty:
        selection = {
            "corrector": "BASE_ONLY", "alpha": 0.0, "robust_score": 0.0,
            "selection_reason": "fallback_no_candidate_passed",
        }
    else:
        best_score = float(eligible["robust_score"].max())
        tied = eligible.loc[
            eligible["robust_score"].ge(best_score - TIE_TOLERANCE)
        ]
        winner = tied.sort_values(
            ["median_gain_pct", "mean_gain_pct", "alpha", "corrector"],
            ascending=[False, False, True, True],
        ).iloc[0]
        selection = winner.to_dict()
        selection["selection_reason"] = "passed_multi_window_rule"
    operational_rows.append({
        **dict(zip(task_columns, keys)), **selection,
        "correction_base_model": CORRECTION_BASE_FAMILY,
        "fallback_policy": FALLBACK_POLICY,
    })

operational_selection = pd.DataFrame(operational_rows)
operational_selection.to_csv(RESULTS_DIR / "10_robust_operational_selection.csv", index=False)
display(operational_selection)
display(operational_selection.groupby(["corrector", "alpha"]).size().rename("tasks").reset_index())


## Final corrector fitting and test predictions

After the operational decisions are frozen, every retained corrector family is refitted on the union of training OOF residuals and validation residuals. Test residual predictions are generated once. The selector never sees test performance.


In [ ]:
test_correction_frames = []
final_training_rows = []

final_progress = tqdm(
    total=len(ACTIVE_RESOLUTIONS) * len(TARGETS) * len(ACTIVE_CORRECTORS),
    desc="Phase D — final corrector refit and test",
    unit="model",
    dynamic_ncols=True,
)

for resolution in ACTIVE_RESOLUTIONS:
    data = datasets[resolution]
    origins = origins_by_resolution[resolution]
    for target in TARGETS:
        bundle = corrector_datasets[(resolution, target)]
        X_test_frame, test_columns, Y_test = build_corrector_dataset(
            data, origins, test_base_predictions, resolution, target
        )
        if bundle["feature_columns"] != test_columns:
            raise AssertionError("Test corrector feature columns differ from training columns.")
        X_test = X_test_frame[test_columns].to_numpy(dtype=float)
        X_final = np.vstack([bundle["X_oof"], bundle["X_validation"]])
        Y_final = np.vstack([bundle["Y_oof"], bundle["Y_validation"]])
        base_test = test_base_predictions.loc[
            test_base_predictions["resolution_minutes"].eq(resolution)
            & test_base_predictions["target"].eq(target)
        ].copy()
        origin_order = np.sort(base_test["origin_index"].unique())

        for corrector in ACTIVE_CORRECTORS:
            final_progress.set_postfix(
                resolution=f"{resolution}min",
                target=target,
                corrector=corrector,
                refresh=True,
            )
            selected = selected_correctors.loc[
                selected_correctors["resolution_minutes"].eq(resolution)
                & selected_correctors["target"].eq(target)
                & selected_correctors["corrector"].eq(corrector)
            ].iloc[0]
            parameters = json.loads(selected["parameters"])
            parameters["candidate_id"] = int(selected["candidate_id"])
            estimator = make_corrector(corrector, parameters)
            started = time.perf_counter()
            estimator.fit(X_final, Y_final)
            fit_seconds = time.perf_counter() - started
            started = time.perf_counter()
            correction = np.asarray(estimator.predict(X_test), dtype=float)
            inference_seconds = time.perf_counter() - started

            model_path = MODEL_DIR / f"{resolution}min" / target / f"{corrector}.joblib"
            model_path.parent.mkdir(parents=True, exist_ok=True)
            joblib.dump({
                "estimator": estimator, "feature_columns": test_columns,
                "parameters": parameters, "horizons_minutes": HORIZONS_MINUTES,
            }, model_path)
            final_training_rows.append({
                "resolution": f"{resolution}min", "resolution_minutes": resolution,
                "target": target, "corrector": corrector,
                "candidate_id": int(selected["candidate_id"]),
                "parameters": selected["parameters"], "n_training_samples": len(X_final),
                "n_test_samples": len(X_test), "fit_seconds": fit_seconds,
                "inference_seconds": inference_seconds,
                "model_path": str(model_path.relative_to(PROJECT_ROOT)),
            })

            correction_wide = pd.DataFrame(correction, index=origin_order, columns=HORIZONS_MINUTES)
            subset = base_test.copy()
            subset["corrector"] = corrector
            subset["residual_prediction"] = [
                correction_wide.at[int(row.origin_index), int(row.horizon_minutes)]
                for row in subset.itertuples(index=False)
            ]
            test_correction_frames.append(subset)
            final_progress.update(1)

final_progress.close()

test_corrector_predictions = pd.concat(test_correction_frames, ignore_index=True)
test_corrector_predictions.to_parquet(
    PREDICTION_DIR / "04_selected_corrector_test_predictions.parquet", index=False
)
pd.DataFrame(final_training_rows).to_csv(RESULTS_DIR / "11_final_corrector_training_summary.csv", index=False)


## Assemble the operational robust strategy

For an activated task, the prediction is

\[
\hat y_{robust}=\hat y_{SARIMA}+\alpha\hat e.
\]

For `BASE_ONLY`, the prediction is taken from notebook 07's selected advanced-traditional model. The SARIMA prediction is retained in the output so gains against the correction base can still be audited.


In [ ]:
def assemble_operational_predictions(split):
    correction_predictions = (
        validation_corrector_predictions if split == "validation" else test_corrector_predictions
    )
    fallback_predictions = (
        fallback_validation_predictions if split == "validation" else fallback_test_predictions
    )
    rows = []
    for decision in operational_selection.itertuples(index=False):
        resolution = int(decision.resolution_minutes)
        target = decision.target
        horizon = int(decision.horizon_minutes)
        base_task = correction_predictions.loc[
            correction_predictions["resolution_minutes"].eq(resolution)
            & correction_predictions["target"].eq(target)
            & correction_predictions["horizon_minutes"].eq(horizon)
        ]
        if decision.corrector == "BASE_ONLY":
            source = fallback_predictions.loc[
                fallback_predictions["resolution_minutes"].eq(resolution)
                & fallback_predictions["target"].eq(target)
                & fallback_predictions["horizon_minutes"].eq(horizon)
            ].copy()
            sarima_lookup = base_task.drop_duplicates("origin_index").set_index("origin_index")["predicted"]
            source["sarima_base_predicted"] = source["origin_index"].map(sarima_lookup)
            source["robust_predicted"] = source["predicted"]
            source["residual_prediction"] = 0.0
            source["operational_model"] = "BASE_ONLY:" + str(
                advanced_selection.loc[
                    advanced_selection["resolution_minutes"].eq(resolution)
                    & advanced_selection["target"].eq(target), "family"
                ].iloc[0]
            )
        else:
            source = base_task.loc[base_task["corrector"].eq(decision.corrector)].copy()
            source["sarima_base_predicted"] = source["predicted"]
            source["robust_predicted"] = (
                source["predicted"] + float(decision.alpha) * source["residual_prediction"]
            )
            source["operational_model"] = (
                CORRECTION_BASE_FAMILY + "+" + decision.corrector + "@alpha=" + str(decision.alpha)
            )
        source["operational_action"] = decision.corrector
        source["alpha"] = float(decision.alpha)
        source["selection_reason"] = decision.selection_reason
        rows.append(source[[
            "resolution", "resolution_minutes", "target", "horizon_minutes", "split",
            "origin_index", "origin_timestamp", "target_timestamp", "observed",
            "sarima_base_predicted", "robust_predicted", "residual_prediction",
            "operational_action", "alpha", "operational_model", "selection_reason",
        ]])
    return pd.concat(rows, ignore_index=True)


robust_validation_predictions = assemble_operational_predictions("validation")
robust_test_predictions = assemble_operational_predictions("test")
robust_validation_predictions.to_parquet(
    PREDICTION_DIR / "05_robust_validation_predictions.parquet", index=False
)
robust_test_predictions.to_parquet(
    PREDICTION_DIR / "06_robust_test_predictions.parquet", index=False
)


In [ ]:
def operational_metrics(predictions, fitting_splits):
    rows = []
    for keys, group in predictions.groupby(
        ["resolution", "resolution_minutes", "target", "horizon_minutes", "operational_action", "alpha", "operational_model"],
        sort=False, observed=True,
    ):
        observed = group["observed"].to_numpy(dtype=float)
        robust = group["robust_predicted"].to_numpy(dtype=float)
        sarima = group["sarima_base_predicted"].to_numpy(dtype=float)
        robust_rmse = mean_squared_error(observed, robust) ** 0.5
        sarima_rmse = mean_squared_error(observed, sarima) ** 0.5
        scale = prediction_scale(datasets[int(keys[1])], keys[2], fitting_splits)
        rows.append({
            "resolution": keys[0], "resolution_minutes": keys[1], "target": keys[2],
            "horizon_minutes": keys[3], "operational_action": keys[4], "alpha": keys[5],
            "operational_model": keys[6], "n": len(group),
            "rmse": robust_rmse, "nrmse": robust_rmse / scale,
            "mae": mean_absolute_error(observed, robust),
            "bias": float(np.mean(robust - observed)), "r2": r2_score(observed, robust),
            "sarima_base_rmse": sarima_rmse,
            "gain_vs_sarima_pct": 100 * (sarima_rmse - robust_rmse) / sarima_rmse,
        })
    return pd.DataFrame(rows)


robust_validation_metrics = operational_metrics(robust_validation_predictions, ["train"])
robust_test_metrics = operational_metrics(robust_test_predictions, ["train", "validation"])
robust_validation_metrics.to_csv(RESULTS_DIR / "12_robust_validation_metrics.csv", index=False)
robust_test_metrics.to_csv(RESULTS_DIR / "13_robust_test_metrics.csv", index=False)
display(robust_test_metrics)


## Historical operational-selection audit

The audited historical selector activated 12 corrections and retained 28 `BASE_ONLY` tasks. The audit below compares the current validation-only decisions with those historical corrector–alpha choices. It does not force a match and never uses test results.


In [ ]:
historical_activated = {
    (12, "relative_humidity", 60): ("XGBOOST", 0.25),
    (12, "relative_humidity", 240): ("XGBOOST", 0.25),
    (12, "temperature", 120): ("HIST_GRADIENT_BOOSTING", 0.25),
    (20, "temperature", 60): ("RANDOM_FOREST", 0.25),
    (30, "relative_humidity", 120): ("HIST_GRADIENT_BOOSTING", 0.25),
    (30, "temperature", 120): ("XGBOOST", 0.25),
    (4, "relative_humidity", 60): ("HIST_GRADIENT_BOOSTING", 0.25),
    (4, "relative_humidity", 120): ("HIST_GRADIENT_BOOSTING", 0.25),
    (4, "relative_humidity", 240): ("EXTRA_TREES", 0.25),
    (4, "relative_humidity", 480): ("XGBOOST", 0.50),
    (4, "temperature", 60): ("HIST_GRADIENT_BOOSTING", 0.25),
    (4, "temperature", 480): ("XGBOOST", 0.25),
}

historical_rows = []
for resolution in RESOLUTIONS:
    for target in TARGETS:
        for horizon in HORIZONS_MINUTES:
            corrector, alpha = historical_activated.get(
                (resolution, target, horizon), ("BASE_ONLY", 0.0)
            )
            historical_rows.append({
                "resolution_minutes": resolution, "target": target,
                "horizon_minutes": horizon, "historical_corrector": corrector,
                "historical_alpha": alpha,
            })
historical_selection = pd.DataFrame(historical_rows)
selection_audit = operational_selection.merge(
    historical_selection, on=["resolution_minutes", "target", "horizon_minutes"], how="left"
)
selection_audit["corrector_match"] = selection_audit["corrector"].eq(selection_audit["historical_corrector"])
selection_audit["alpha_match"] = np.isclose(selection_audit["alpha"], selection_audit["historical_alpha"])
selection_audit["complete_match"] = selection_audit["corrector_match"] & selection_audit["alpha_match"]
selection_audit.to_csv(RESULTS_DIR / "14_historical_operational_selection_audit.csv", index=False)
display(selection_audit)


## Diagnostic figure and execution report


In [ ]:
figure_data = robust_test_metrics.copy()
figure_data["rmse_difference_robust_minus_sarima"] = (
    figure_data["rmse"] - figure_data["sarima_base_rmse"]
)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for axis, target in zip(axes, TARGETS):
    subset = figure_data.loc[figure_data["target"].eq(target)]
    sns.lineplot(
        data=subset, x="horizon_minutes", y="rmse_difference_robust_minus_sarima",
        hue="resolution", markers=True, dashes=False, ax=axis,
    )
    axis.axhline(0, color="black", linewidth=1)
    axis.set_title(target.replace("_", " ").title())
    axis.set_xlabel("Forecast horizon (min)")
    axis.set_ylabel("RMSE difference: robust − SARIMA")
figure_path = FIGURE_DIR / "08_robust_minus_sarima_rmse.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

report = {
    "notebook": "08_robust_residual_hybrid.ipynb",
    "execution_mode": EXECUTION_MODE,
    "correction_base_family": CORRECTION_BASE_FAMILY,
    "fallback_policy": FALLBACK_POLICY,
    "oof_prediction_rows": len(oof_base_predictions),
    "validation_origins": int(base_validation_predictions["origin_index"].nunique()),
    "corrector_tuning_rows": len(corrector_tuning),
    "window_candidate_metric_rows": len(window_candidate_metrics),
    "candidate_robustness_rows": len(candidate_robustness),
    "operational_tasks": len(operational_selection),
    "activated_corrections": int(operational_selection["corrector"].ne("BASE_ONLY").sum()),
    "base_only_tasks": int(operational_selection["corrector"].eq("BASE_ONLY").sum()),
    "robust_validation_prediction_rows": len(robust_validation_predictions),
    "robust_test_prediction_rows": len(robust_test_predictions),
    "historical_selection_matches": int(selection_audit["complete_match"].sum()),
    "test_used_for_selection": False,
}
(METADATA_DIR / "08_robust_residual_hybrid_report.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)
display(pd.DataFrame({"item": report.keys(), "value": report.values()}))
print(f"Results: {RESULTS_DIR.relative_to(PROJECT_ROOT)}")
print(f"Models: {MODEL_DIR.relative_to(PROJECT_ROOT)}")
print(f"Figure: {figure_path.relative_to(PROJECT_ROOT)}")


## Reproducibility note

- Use only `EXECUTION_MODE = "full"` for manuscript results.
- Run notebook 07 in full mode before this notebook.
- Do not use test metrics to alter a corrector, alpha, eligibility threshold, or fallback decision.
- Interpret `BASE_ONLY` as a deliberate validation-based conservative decision, not as a failed model fit.
- Keep the correction-base and fallback definitions distinct in the manuscript and downstream benchmark.
- Notebook `09_final_model_family_benchmark.ipynb` must consume the frozen operational predictions produced here.
